# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

**Deel: Opdracht 2 — Time series feature engineering** · Ahmed Bashabeeb

Dit notebook is het tweede deel van ons portfolio. Aan het eind worden de delen samengevoegd in
`Portfolio2_CityBike.ipynb`. De data (`train.csv`, `test.csv`) staat in de map `data/`. Het
inladen en opschonen van de data gebeurt met de code uit Opdracht 1 (`opdracht1_eda.ipynb`).

**Inhoud**

- Voorbereiding: imports, instellingen en data uit Opdracht 1
- 2.1 Evaluatiemethode · 2.2 Stationariteit (Dickey-Fuller) · 2.3 Trend
- 2.4 Seizoenspatronen met Fourier-analyse · 2.5 Cycli, autocorrelatie en lags
- 2.6 Overige tijdgebaseerde features · 2.7 Klasse `FeatureEngineer`
- 2.8 Invloed van de features · 2.9 Gekozen features

## Voorbereiding: imports, instellingen en data uit Opdracht 1

We gebruiken de klasse `DataLoader` (paragraaf 1.2) en de functie `add_calendar_features`
(paragraaf 1.5) uit Opdracht 1, en schonen de data op zoals in paragraaf 1.9: `hum == 0`
vervangen door interpolatie en 29–30 oktober 2012 verwijderen.

In [ ]:
# Bron: Week 10 – Regression Time Series 2 (import-cel)
# Bron: Week 11 – Time series analysis 3 (imports statsmodels)

# Importeren van de packages die nodig zijn voor de feature engineering
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.fft import fft
from scipy.signal import find_peaks
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.deterministic import (CalendarFourier,
                                           DeterministicProcess)
from statsmodels.tsa.stattools import adfuller
from xgboost import XGBRegressor

# Waarschuwingen onderdrukken voor een overzichtelijk notebook
warnings.filterwarnings('ignore')

# Vaste instellingen
DATA_FOLDER = os.path.join('..', 'data')     # map met de Kaggle-bestanden
RANDOM_STATE = 42                            # reproduceerbare resultaten
HORIZON = 19 * 24                            # 456 uur, zoals de Kaggle-test
HOLDOUT_START = pd.Timestamp('2012-11-12')   # laatste 19 dagen = testset
pd.set_option('display.max_colwidth', 150)
plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

In [ ]:
# Bron: Opdracht 1, paragraaf 1.2 (klasse DataLoader), 1.5
#       (add_calendar_features) en 1.9 (opschonen); opdracht1_eda.ipynb

class DataLoader:
    """Leest de bestanden van de Kaggle-competitie in en schoont ze op."""

    def __init__(self, folder='.'):
        self.folder = folder

    def load(self, file_name):
        """Lees een CSV-bestand in met 'date_hour' als datetime."""
        path = os.path.join(self.folder, file_name)
        return pd.read_csv(path, parse_dates=['date_hour'])

    @staticmethod
    def clean(data, outlier_days):
        """Schoon de data op (onderbouwing: paragraaf 1.4 en 1.9).

        - sorteren op tijd en dubbele uren verwijderen;
        - luchtvochtigheid 0 (meetfout) vervangen door interpolatie;
        - uitzonderlijke dagen (outliers) verwijderen.
        """
        data = data.sort_values('date_hour')
        data = data.drop_duplicates('date_hour').copy()
        data.loc[data['hum'] == 0, 'hum'] = np.nan
        data['hum'] = data['hum'].interpolate()
        days = data['date_hour'].dt.normalize()
        data = data[~days.isin(pd.to_datetime(outlier_days))]
        return data.reset_index(drop=True)


def add_calendar_features(data):
    """Voeg tijdselementen toe die uit 'date_hour' volgen."""
    data = data.copy()
    time = data['date_hour']
    data['hour'] = time.dt.hour
    data['weekday'] = time.dt.dayofweek           # 0 = maandag, 6 = zondag
    data['day_name'] = time.dt.day_name()
    data['week'] = time.dt.isocalendar().week.astype(int)
    data['month'] = time.dt.month
    data['year'] = time.dt.year - 2011            # 0 = 2011, 1 = 2012
    data['dayofyear'] = time.dt.dayofyear
    # werkdag = maandag t/m vrijdag en geen feestdag
    is_weekday = data['weekday'] < 5
    data['workingday'] = (is_weekday & (data['holiday'] == 0)).astype(int)
    return data


# Data inladen en opschonen zoals in Opdracht 1 (paragraaf 1.9)
OUTLIER_DAYS = ['2012-10-29', '2012-10-30']
loader = DataLoader(DATA_FOLDER)
train = loader.clean(loader.load('train.csv'), OUTLIER_DAYS)
test = loader.clean(loader.load('test.csv'), outlier_days=[])
print(f'train: {len(train)} rijen | test: {len(test)} rijen')

# 2. Opdracht 2: Time series feature engineering

In deze opdracht onderzoeken we stationariteit, trend, seizoenspatronen en cycli, maken we daarvan
features en meten we wat hun invloed is op de voorspellingen. Dit was een **iteratief proces**:
features toevoegen → model testen → nieuwe features proberen (2.8).

## 2.1 Evaluatiemethode

Om de invloed van features en modellen eerlijk te meten, leggen we eerst de evaluatie vast.

- **Niet husselen.** Bij tijdreeksen mag je de data niet willekeurig verdelen: dan traint het model
  op de toekomst en wordt de score te optimistisch (*data leakage*). Cross-validatie bij tijdreeksen
  werkt daarom anders dan gewone k-fold (notebook *Cross validation*; De Haagse Hogeschool,
  z.d.-a): elke fold traint op het **verleden** en valideert op de periode **erna**
  (`TimeSeriesSplit`; opdracht 11 van *Regression Time Series 2*, De Haagse Hogeschool, z.d.-d;
  *Week 11 application remarks*, De Haagse Hogeschool, z.d.-j; Holdgraf, z.d.).
- **Trainset en testset.** Net als in opdracht 3.1 van *Regression Time Series 1* en de klasse
  `RegressionModels` uit *Regression Time Series 3* (`train_test_split(shuffle=False)`; De Haagse
  Hogeschool, z.d.-c, z.d.-e) houden we de **laatste periode**, 19 dagen (12–30 november 2012),
  apart als **testset**. Die gebruiken we pas aan het eind, één keer per model. De hyperparameters
  kiezen we met cross-validatie op de trainset. Daarna trainen we het beste model opnieuw op álle
  data voor Kaggle (Radhakrishnan, 2023a).
- **Folds op de trainset.** Elke validatieperiode is **456 uur (19 dagen)**, net zo lang als de
  Kaggle-test:
    - 4 **herfst-folds** (`TimeSeriesSplit(n_splits=4, test_size=456)`): de laatste 2,5 maand vóór de
      testset;
    - 3 **winter-folds**: 1–19 december 2011, 9–27 januari 2012 en 6–24 februari 2012.

  Meer folds geven een betrouwbaarder gemiddelde, maar kosten meer rekentijd; met 7 folds zitten
  beide seizoenen er goed in.
- **Waarom winter-folds? (iteratie na Kaggle-feedback)** In onze eerste versie gebruikten we alleen
  herfst-folds. Op Kaggle bleek dat modellen op log-schaal (lineaire regressie, SARIMAX) veel beter
  scoorden dan onze validatie voorspelde (zie 4.3). December heeft lage aantallen en gedraagt zich
  anders dan de drukke herfst. Daarom valideren we nu ook op winterperiodes. De code voor de
  winter-folds komt niet in de lesstof voor (Claude, 2026. Prompt 2: Winter-folds voor CV).

**Metrics** (opdracht 2 van *Regression 1*; De Haagse Hogeschool, z.d.-b): $R^2$ (verklaarde
variantie), MAE (gemiddelde absolute fout),
MSE (gemiddelde gekwadrateerde fout) en **RMSE** $=\sqrt{\text{MSE}}$, de maatstaf van Kaggle:
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$
De RMSE heeft dezelfde eenheid als `cnt` (fietsen per uur) en straft grote fouten zwaar af.

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 11 (TimeSeriesSplit)
# Bron: Week 11 – application remarks (TimeSeriesSplit)
# Bron: Week 8 – Regression 1, opdracht 2 (R2, MAE, MSE en RMSE)
# Bron: Week 8 – Cross validation (cross_val_score)
# Bron: Week 9 – Regression Time Series 1, opdracht 3.1 (testset)
# Prompt 2: Winter-folds voor CV

WINTER_STARTS = ['2011-12-01', '2012-01-09', '2012-02-06']
N_AUTUMN = 4


def make_folds(data, n_autumn=N_AUTUMN, winter_starts=WINTER_STARTS):
    """Maak de folds: (trainingsrijen, validatierijen) per fold.

    Elke fold traint op alle rijen vóór de validatieperiode van 456 uur.
    """
    autumn = TimeSeriesSplit(n_splits=n_autumn, test_size=HORIZON)
    folds = list(autumn.split(data))
    for start in pd.to_datetime(winter_starts):
        end = start + pd.Timedelta(hours=HORIZON)
        train_idx = np.where(data['date_hour'] < start)[0]
        val_idx = np.where((data['date_hour'] >= start)
                           & (data['date_hour'] < end))[0]
        folds.append((train_idx, val_idx))
    return folds


def rmse(y_true, y_pred):
    """Root Mean Squared Error (de maatstaf van Kaggle)."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def regression_metrics(y_true, y_pred):
    """R2, MAE, MSE en RMSE (zoals in de les Regression 1)."""
    mse = mean_squared_error(y_true, y_pred)
    return {'R2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mse, 'RMSE': np.sqrt(mse)}


# Trainset en testset: de laatste 19 dagen zijn de testset (holdout)
train_part = train[train['date_hour'] < HOLDOUT_START].reset_index(drop=True)
holdout = train[train['date_hour'] >= HOLDOUT_START].reset_index(drop=True)
CV = make_folds(train_part)


def cv_rmse(model, X, y):
    """Gemiddelde RMSE over alle folds van de trainset."""
    scores = cross_val_score(model, X, y, cv=CV,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()


fold_overview = []
for fold, (train_idx, val_idx) in enumerate(CV, start=1):
    fold_overview.append({
        'fold': fold,
        'type': 'herfst' if fold <= N_AUTUMN else 'winter',
        'training t/m': train_part['date_hour'][train_idx].max(),
        'validatie van': train_part['date_hour'][val_idx].min(),
        'validatie t/m': train_part['date_hour'][val_idx].max()})
print(f'trainset: {len(train_part)} uur | testset: '
      f'{holdout["date_hour"].min():%d-%m-%Y} t/m '
      f'{holdout["date_hour"].max():%d-%m-%Y} ({len(holdout)} uur)')
pd.DataFrame(fold_overview)